# $Bx$ component as Ito's process 
This notebook is devoted to analysis of connection between $a(t)$ and $b(t)$ 
from Ito equation: 
$$dX = a(t)dt + b(t)dW, \quad \text{where } W \text { is a normal Wiener process}$$

In [ ]:
# Import modules
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as ple
import plotly.subplots as sp
from plotly.offline import init_notebook_mode
from IPython.display import display, HTML
import pickle
from tqdm.notebook import tqdm


# Allows you to use modified modules without rebooting the kernel
%load_ext autoreload
%autoreload 2
# Enable latex on plotly figures
init_notebook_mode()
display(
    HTML(
        '<script type="text/javascript" async src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.1/MathJax.js?config=TeX-MML-AM_SVG"></script>'
    )
)

In [ ]:
def convert_dateticks(dates):
    return pd.date_range(start=dates[0], end=dates[-1], freq="D")


# Set default layout settings
def set_default_layout(fig: go.Figure, ticks_size=14, labels_size=18, dates=[]):
    fig.update_layout(
        xaxis=dict(
            tickmode="linear",
            tickformat="%d-%m-%Y",
            tickvals=convert_dateticks(dates),
            tickfont=dict(size=ticks_size),
        ),
        yaxis=dict(tickfont=dict(size=ticks_size)),
    )
    fig.update_xaxes(
        tickangle=90,
        title_text="Дата",
        title_font={"size": labels_size},
        # title_standoff = 25
    )

    fig.update_yaxes(
        title_text=" Значения корреляции",
        title_font={"size": labels_size},
        # title_standoff = 25
    )

In [ ]:
import os

out_dir = "/home/oplora/Desktop/MagfieldFigures116"
# Create dir if it doesn't exist
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

figure_size = dict(height=600, width=1000)
axis_font_size = 20
axis_title_font_size = 24
colors = ["#000000", "#FF7F50"]

In [ ]:
# Load up gaussian mixture model for component Bx
series_name = "Bz"
# with open(f"data/d{series_name}_4300_4.pkl", "rb") as f:
with open(f"data/d{series_name}_4320_3.pkl", "rb") as f:
    gmm = pickle.load(f)

## Processes Bx, By, Bz

In [ ]:
dataset = pd.read_csv("../../src/datasets/2020_ydhm_id.csv")
dataset["ydhm_id"] = pd.to_datetime(dataset["ydhm_id"])
print(dataset.dtypes)
dataset = dataset[dataset["ydhm_id"].apply(lambda x: x.month) == 3]
dataset.dropna(inplace=True)

In [ ]:
import plotly.io as pio

# Set default theme to dark or light globally
pio.templates.default = (
    "plotly_white"  # Options include: 'plotly_white', 'ggplot2', 'seaborn'
)
# 1. Get the current active template (usually "plotly")
current_template = pio.templates[pio.templates.default]

# 2. Update grid colors globally for both X and Y axes
gridColor = "#464646"
current_template.layout.xaxis.gridcolor = gridColor
current_template.layout.yaxis.gridcolor = gridColor

# 3. (Optional) Update zero lines to match your grid style
current_template.layout.xaxis.zerolinecolor = gridColor
current_template.layout.yaxis.zerolinecolor = gridColor

In [ ]:
fig = ple.line(
    dataset, x="ydhm_id", y="{0}".format(series_name), color_discrete_sequence=["black"]
)

fig.update_layout(
    height=figure_size["height"],
    width=figure_size["width"],
    showlegend=False,
)

fig.update_xaxes(
    tickmode="linear",
    tickformat="%d-%m-%Y",
    tickvals=convert_dateticks(dataset["ydhm_id"].values),
    tickfont=dict(size=axis_font_size),
    dtick=pd.Timedelta("3 days").total_seconds() * 1000,
    tickangle=90,
    title_text="Дата",
    title_font={"size": axis_title_font_size},
)

fig.update_yaxes(
    tickfont=dict(size=axis_font_size),
    title_text="Значение, нТ",
    title_font={"size": axis_title_font_size},
)

fig.show()
# fig.write_image(f"{out_dir}/{series_name.capitalize()}_march.jpg", format="jpg")
# del dataset

## Calculate $a(t), \space b(t)$

Parameters of Ito's equation are calculated in the next way:

$$
a(t) = \sum_{k=1}^{K}{p_k a_k}, \quad 
b(t) = \sum_{k=1}^{K}{p_k b_k},
$$
where $K$ is a number of mixture components

In [ ]:
from magfield.em.auxiliary import smooth

p = gmm["weights"]
a = gmm["means"]
b = gmm["variances"]
K = gmm["num_comp"]
nc = gmm["num_comp"]

# smoothcoef = False
smoothcoef = True

coef_a = np.sum(p * a, axis=0)
coef_b = np.sum(p * b, axis=0)

if smoothcoef:
    # smooth_N = int(0.05 * len(coef_a))
    smooth_N = 60 * 12  # * 7
    coef_a = smooth(coef_a, smooth_N)
    coef_b = smooth(coef_b, smooth_N)

limit = sum(gmm["dates"] < pd.to_datetime("2020-02-01 00:00:00"))

In [ ]:
# Create subplots
# fig = sp.make_subplots(rows=2, cols=1,
#                     subplot_titles=(rf"Ito's coefficient $a(t) = \sum_{{k=1}}^{K}{{p_k a_k}}$",
#                                     rf"Ito's coefficient $b(t) = \sum_{{k=1}}^{K}{{p_k b_k}}$"))
fig = sp.make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.1,
)

x = gmm["dates"][:limit]
# Add traces for the first plot (coef_a)
fig.add_trace(
    go.Scatter(
        x=x, y=coef_a[:limit], mode="lines", name="coef_a", line=dict(color="black")
    ),
    row=1,
    col=1,
)
# Add traces for the second plot (coef_b)
fig.add_trace(
    go.Scatter(
        x=x, y=coef_b[:limit], mode="lines", name="coef_b", line=dict(color="black")
    ),
    row=2,
    col=1,
)

# Update layout for better presentation (optional)
smooth_title = ""
smooth_file = ""
if smoothcoef:
    smooth_title = f" smoothed by {smooth_N} minutes."
    smooth_file = f"_smooth_{smooth_N}"
# fig.update_layout(height=600,width=1800, title_text=f"Ito's Coefficients for d{series_name}" + smooth_title, showlegend=False)

# # Update the font size of x-axis tick labels for all subplots
# fig.update_xaxes(tickfont=dict(size=axis_font_size))
# # Update the font size of y-axis tick labels for all subplots
# fig.update_yaxes(tickfont=dict(size=axis_font_size))

fig.update_layout(
    height=figure_size["height"], width=figure_size["width"], showlegend=False
)
fig.update_xaxes(
    tickmode="linear",
    tickformat="%d-%m",
    tickvals=convert_dateticks(gmm["dates"][:limit]),
    tickfont=dict(size=axis_font_size),
    # title_standoff = 25
)
fig.update_xaxes(
    tickmode="linear",
    tick0=x[0],  # Optional: Starting point anchor for the ticks
    dtick=86400000 * 6,  # Forces a tick every 2 days (2 * 1 day in ms)
    tickangle=45,
    title_text="Дата",
    title_font={"size": axis_title_font_size},
    row=2,
    col=1,
)

fig.update_yaxes(
    tickfont=dict(size=axis_font_size),
    # title_standoff = 25
)

fig.add_annotation(
    text="Значение, нТл",
    xref="paper",
    yref="paper",
    x=-0.06,
    y=0.5,  # Adjust x and y for desired position
    showarrow=False,
    textangle=-90,  # Rotate for vertical title
    xanchor="right",
    yanchor="middle",
    font=dict(size=axis_title_font_size),
)

# Show the plot
# fig.show()
fig.write_image(f"{out_dir}/d{series_name}_Ito_coeffs{smooth_file}.png")

# Correlation plots

In [ ]:
correlation_window = (gmm["window"]["size"], gmm["window"]["step"])

#### Correlation $a(t)$, $b(t)=\sum_{j=1}^Kp_jb_j$

In [ ]:
correlation = []
corr2 = []  # Just for plot (cell below)
for i in tqdm(range(0, len(coef_a) - correlation_window[0], correlation_window[1])):
    sub_a = coef_a[i : correlation_window[0] + i]
    sub_b = coef_b[i : correlation_window[0] + i]
    correlation.append(np.corrcoef(sub_a, sub_b)[0, 1])
    corr2.append(np.corrcoef(np.abs(sub_a), sub_b)[0, 1])

#### Correlation plots

In [ ]:
# fig = ple.line(
#     x=gmm["dates"][:limit], y=correlation[:limit], color_discrete_sequence=[colors[0]]
# )

# fig.update_layout(
#     height=figure_size["height"],
#     width=figure_size["width"],
#     showlegend=False,
# )

# fig.update_xaxes(
#     tickmode="linear",
#     tickformat="%d-%m",
#     tickvals=convert_dateticks(gmm["dates"][:limit]),
#     tickfont=dict(size=axis_font_size),
#     dtick=pd.Timedelta("5 days").total_seconds() * 1000,
#     tickangle=90,
#     title_text="Дата",
#     title_font={"size": axis_title_font_size},
# )
# fig.update_yaxes(
#     tickfont=dict(size=axis_font_size),
#     title_text="Значение, нТ",
#     title_font={"size": axis_title_font_size},
# )


# fig.write_image(
#     f"{out_dir}/d{series_name}_corr{nc}_{correlation_window[0]}{smooth_file}_1.png"
# )

# fig = ple.line(
#     x=gmm["dates"][:limit], y=corr2[:limit], color_discrete_sequence=[colors[0]]
# )

# fig.update_layout(
#     height=figure_size["height"],
#     width=figure_size["width"],
#     showlegend=False,
# )

# fig.update_xaxes(
#     tickmode="linear",
#     tickformat="%d-%m",
#     tickvals=convert_dateticks(gmm["dates"][:limit]),
#     tickfont=dict(size=axis_font_size),
#     dtick=pd.Timedelta("5 days").total_seconds() * 1000,
#     tickangle=90,
#     title_text="Дата",
#     title_font={"size": axis_title_font_size},
# )
# fig.update_yaxes(
#     tickfont=dict(size=axis_font_size),
#     title_text="Значение, нТ",
#     title_font={"size": axis_title_font_size},
# )


# fig.write_image(
#     f"{out_dir}/d{series_name}_corr{nc}_{correlation_window[0]}{smooth_file}_2.png"
# )
# del corr2

#### Correlation plots for hole time span

In [ ]:
from plotly.subplots import make_subplots


def long_plot(series, title, n, dates, plotHight=300, figWidth=1200, showlegend=True):
    """Represent long time series.
    Slice it down into `n` sections and put one under each other.
    :param series: 1D data to visualize.
    :param dates: Custom units for X-axis.
    :param title: Custom name of the figure.
    :param n: Number of subplots that would be one figure. Each of them
    represent one n'th of original series."""

    fig = make_subplots(rows=n, cols=1, vertical_spacing=0.03)

    # Calculate the length of each partition
    total_length = len(series)
    partition_length = total_length // n

    # Create each part and add to the figure
    for i in range(n):
        start_index = i * partition_length
        # To handle the last segment which may include extra elements
        end_index = (i + 1) * partition_length if (i + 1) < n else total_length

        # Slice the series data for this part
        part = series[start_index:end_index]

        # Add the plot for the current part
        fig.append_trace(
            go.Scatter(
                x=dates[start_index:end_index],  # Use the corresponding dates
                y=part,
                name=f"Part {i + 1}",
                line=dict(color="black"),
            ),
            row=i + 1,
            col=1,
        )
        fig.update_xaxes(
            # tickangle=90,
            tickmode="linear",
            tickformat="%d-%m",
            tickvals=convert_dateticks(gmm["dates"][:limit]),
            tickfont=dict(size=axis_font_size),
            dtick=pd.Timedelta("5 days").total_seconds() * 1000,
        )
        fig.update_yaxes(
            tickfont=dict(size=axis_font_size),
        )

    fig.update_xaxes(
        title_text="Дата",
        title_font={"size": axis_title_font_size},
        row=n,
        col=1,
    )

    fig.add_annotation(
        text="Значение корреляции",
        xref="paper",
        yref="paper",
        x=-0.06,
        y=0.5,  # Adjust x and y for desired position
        showarrow=False,
        textangle=-90,  # Rotate for vertical title
        xanchor="right",
        yanchor="middle",
        font=dict(size=axis_title_font_size),
    )
    # Update the layout of the figure
    fig.update_layout(
        height=plotHight * n, width=figWidth, title_text=title, showlegend=showlegend
    )

    return fig

In [ ]:
# title = rf"""$\text{{Correlation between }} a(t), b(t)=
#     \sum_{{j=1}}^{gmm["num_comp"]}p_jb_j \text{{ on window size }}
#     {correlation_window[0]} \text{{ minutes}}$
#     """
# cp = long_plot(correlation, title, dates=gmm["dates"], n=12)
cp = long_plot(
    correlation,
    "",
    dates=gmm["dates"],
    n=6,
    plotHight=200,
    figWidth=figure_size["width"],
    showlegend=False,
)
# cp.show()
cp.write_image(
    f"{out_dir}/d{series_name}_corr{nc}_{correlation_window[0]}{smooth_file}.png"
)

#### Histogram of correlation between a(t) and b(t)

In [ ]:
# # Create a histogram plot for a correlation of coef_a and coef_b using plolty express

# percent = np.count_nonzero(np.array([abs(x) for x in correlation]) > 0.5) / len(
#     correlation
# )
# fig = ple.histogram(
#     x=correlation,
#     nbins=10,
#     # labels={"x": "Correlation", "y": "percents"},
#     # title=f"Percent of values higher than 0.5 for correlation a(t), b(t) of d{series_name} - {round(percent,3)}",
#     # title=f"Процент значений больших 0.5 по абсолютному значению - {round(percent,3)}",
#     histnorm="percent",
#     # labels={"x": "Значения корреляции", "y": "Процент значений"},
#     height=figure_size["height"],
#     width=figure_size["width"],
#     color_discrete_sequence=["black"],
# )
# fig.update_xaxes(
#     # tickangle=90,
#     title_text="Значения корреляции",
#     title_font={"size": axis_title_font_size},
#     tickmode="linear",
#     tickfont=dict(size=axis_font_size),
#     dtick=0.1,
# )
# fig.update_yaxes(
#     tickfont=dict(size=axis_font_size),
#     title_text="Процент значений",
#     title_font={"size": axis_title_font_size},
# )
# fig.update_layout(bargap=0.2)

# # fig.show()
# fig.write_image(
#     f"{out_dir}/d{series_name}_histogram_corr{nc}_{correlation_window[0]}{smooth_file}.png"
# )

# Analysis of $a(t)$ and $b(t)$ relation

## Trigonometric approximation

### Static harmonics for all data

In [ ]:
hn = 24

In [ ]:
import numpy as np
from scipy.optimize import leastsq
import plotly.graph_objects as go
from scipy.stats import norm


def find_frequency(data, sampling_rate):
    """"""
    n = len(data)
    fft_result = np.fft.fft(data)
    freqs = np.fft.fftfreq(n, d=1 / sampling_rate)
    spectrum = abs(fft_result)
    idx = np.argmax(spectrum[1:]) + 1  # Set initial frequency
    freq = freqs[idx]
    return abs(freq)


def harmonic_approximation(
    data, time, harmonics_num=4, smooth=0, title="", limit=-1, figHeight=600
):
    if smooth:

        def smoothing(y, box_pts):
            box = np.ones(box_pts) / box_pts
            y_smooth = np.convolve(y, box, mode="same")
            return y_smooth

        data = smoothing(data, smooth)

    fig = make_subplots(rows=3, cols=1, vertical_spacing=0.14)
    # fig = make_subplots(rows=2, cols=1)
    data_orig = data
    t = np.array(range(len(data)))
    params = []
    harmonical_signal = np.zeros(len(data))

    for _ in range(harmonics_num):
        # guess_mean = np.mean(data)
        guess_phase = 0
        guess_freq = find_frequency(data, len(data)) * np.pi * 2 / len(data)
        guess_amp = max(data) - min(data)

        def optimize_func(x):
            # return x[0] * np.sin(x[1] * t + x[2]) + x[3] - data
            return x[0] * np.sin(x[1] * t + x[2]) - data

        params_sin = leastsq(
            # optimize_func, [guess_amp, guess_freq, guess_phase, guess_mean]
            optimize_func,
            [guess_amp, guess_freq, guess_phase],
        )[0]

        params.append(params_sin)
        # est_amp, est_freq, est_phase, est_mean = params_sin
        est_amp, est_freq, est_phase = params_sin

        # data_fit = est_amp * np.sin(est_freq * t + est_phase) + est_mean
        data_fit = est_amp * np.sin(est_freq * t + est_phase)
        data = data - data_fit
        harmonical_signal += data_fit

    # APPROXIMATION VISUALIZATION
    fig.add_trace(
        go.Scatter(x=time[:limit], y=data_orig[:limit], marker=dict(color=colors[0])),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=time[:limit], y=harmonical_signal[:limit], marker=dict(color=colors[1])
        ),
        row=1,
        col=1,
    )
    # fig.add_annotation(
    #     xref="x domain",
    #     yref="y domain",
    #     x=0.5,
    #     y=1.2,
    #     showarrow=False,
    #     font=dict(size=22),
    #     text=f"<b>Approximation ({harmonics_num} harmonics)<b>",
    #     row=1,
    #     col=1,
    # )
    #
    fig.update_xaxes(
        tickangle=90,
        tickmode="linear",
        tickformat="%d-%m",
        tickvals=convert_dateticks(time[:limit]),
        tickfont=dict(size=axis_font_size),
        dtick=pd.Timedelta("15 days").total_seconds() * 1000,
        row=1,
        col=1,
    )

    # RESIDUALS
    fig.add_trace(
        go.Scatter(x=time[:limit], y=data[:limit], marker=dict(color=colors[0])),
        row=2,
        col=1,
    )
    fig.update_xaxes(
        tickangle=90,
        title_text="Дата",
        title_font={"size": axis_title_font_size},
        tickmode="linear",
        tickformat="%d-%m",
        tickvals=convert_dateticks(time[:limit]),
        tickfont=dict(size=axis_font_size),
        dtick=pd.Timedelta("15 days").total_seconds() * 1000,
        row=2,
        col=1,
    )
    # fig.add_annotation(
    #     xref="x domain",
    #     yref="y domain",
    #     x=0.5,
    #     y=1.2,
    #     showarrow=False,
    #     font=dict(size=22),
    #     text="<b>Residuals<b>",
    #     row=2,
    #     col=1,
    # )

    fig.add_annotation(
        text="Значение, нТл",
        xref="paper",
        yref="paper",
        x=-0.06,
        y=0.66,  # Adjust x and y for desired position
        showarrow=False,
        textangle=-90,  # Rotate for vertical title
        xanchor="right",
        yanchor="middle",
        font=dict(size=axis_title_font_size),
    )
    # HISTOGRAM
    fig.add_annotation(
        text="Пл-ть распр-я, 1/нТл",
        xref="paper",
        yref="paper",
        x=-0.06,
        y=0.13,  # Adjust x and y for desired position
        showarrow=False,
        textangle=-90,  # Rotate for vertical title
        xanchor="right",
        yanchor="middle",
        font=dict(size=axis_title_font_size),
    )
    fig.add_trace(
        go.Histogram(
            x=data[:limit], histnorm="probability density", marker=dict(color=colors[0])
        ),
        row=3,
        col=1,
    )
    fig.update_xaxes(
        # tickangle=90,
        title_text="Отклонение, нТл",
        title_font={"size": axis_title_font_size},
        tickfont=dict(size=axis_font_size),
        row=3,
        col=1,
    )
    x = np.linspace(min(data) * 1.2, max(data) * 1.2, 100)
    fig.add_trace(
        go.Scatter(
            x=x[:limit],
            y=norm.pdf(x[:limit], *norm.fit(data[:limit])),
            marker=dict(color=colors[1]),
        ),
        row=3,
        col=1,
    )

    def cdf(x):
        return norm.cdf(x, loc=norm.fit(data)[0], scale=norm.fit(data)[1])

    # pval = kstest(data, cdf=cdf).pvalue
    # fig.add_annotation(
    #     xref="x domain",
    #     yref="y domain",
    #     x=0.5,
    #     y=1.3,
    #     showarrow=False,
    #     font=dict(size=22),
    #     text="<b>Residuals histogram<b>",
    #     row=3,
    #     col=1,
    # )
    # fig.add_annotation(
    #     xref="x domain",
    #     yref="y domain",
    #     x=0.5,
    #     y=1.165,
    #     showarrow=False,
    #     font=dict(size=20),
    #     text=f"Normal test p-value = {pval:.3f}",
    #     row=3,
    #     col=1,
    # )

    fig.update_yaxes(
        tickangle=-90,
        tickfont=dict(size=axis_font_size),
    )
    fig.update_layout(
        width=figure_size["width"],
        height=figHeight,
        # title=dict(font=dict(size=20), text=f"<b>{title}<b>"),
        showlegend=False,
    )
    return fig, data, params

In [ ]:
fig, resids, paramA = harmonic_approximation(
    data=coef_a,
    time=gmm["dates"][: len(correlation)],
    harmonics_num=hn,
    figHeight=figure_size["height"] + 150,
    # title=f"d{series_name} - a(t)",
    # limit=limit*3
)
# fig.show()
# fig.write_image(
#     f"{out_dir}/d{series_name}_harmonic_approx_a_{hn}_{correlation_window[0]}{smooth_file}.png"
# )

In [ ]:
fig, resids, paramB = harmonic_approximation(
    # data=correlation,
    data=coef_b,
    time=gmm["dates"][: len(correlation)],
    harmonics_num=hn,
    # title=f"d{series_name} - b(t)",
)
# # fig.show()
# # fig.write_image(f"{out_dir}/d{series_name}_harmonic_approx_b_{hn}_{correlation_window[0]}{smooth_file}.png")

In [ ]:
# fig, resids, paramC = harmonic_approximation(
#     data=correlation,
#     time=gmm["dates"][: len(correlation)],
#     harmonics_num=hn,
#     # title=f"d{series_name} - correlation a(t), b(t)",
# )
# # fig.show()
# # fig.write_image(f"{out_dir}/d{series_name}_harmonic_approx_corr_{hn}_{correlation_window[0]}{smooth_file}.png")

### Parameters of harmonics

In [ ]:
from sys import stdout

harmonicsA = pd.DataFrame(paramA, columns=["Amplitude", "Frequency", "Phase"])
harmonicsA["Frequency"] = harmonicsA["Frequency"] * (60 * 24)  # convert to days
harmonicsA["Period"] = 1 / harmonicsA["Frequency"]
harmonicsA["Amplitude"] = harmonicsA["Amplitude"] * 1000  # convert to pico-Tesla
harmonicsA = harmonicsA.reindex(columns=["Amplitude", "Frequency", "Period", "Phase"])
# display(harmonicsA)
harmonicsA.to_latex(stdout, float_format="%.3f")

# Forecasting

In [ ]:
train_size = int(len(correlation) * 0.95)

In [ ]:
fig, resids, paramA = harmonic_approximation(
    data=coef_a,
    time=gmm["dates"][:train_size],
    harmonics_num=hn,
    # title=f"d{series_name} - a(t)",
)

In [ ]:
fig, resids, paramB = harmonic_approximation(
    # data=correlation,
    data=coef_b,
    time=gmm["dates"][:train_size],
    harmonics_num=hn,
    # title=f"d{series_name} - b(t)",
)

In [ ]:
fig, resids, paramC = harmonic_approximation(
    data=correlation,
    time=gmm["dates"][:train_size],
    harmonics_num=hn,
    # title=f"d{series_name} - correlation a(t), b(t)",
)

In [ ]:
test_data_a = coef_a[train_size + 1 : len(correlation)]
test_data_b = coef_b[train_size + 1 : len(correlation)]
test_data_corr = correlation[train_size + 1 : len(correlation)]

sinA, sinB, sinC = [], [], []
for t in tqdm(range(train_size + 1, len(correlation))):
    sa, sb, sc = 0, 0, 0
    for harm in range(hn):
        sa += paramA[harm][0] * np.sin(paramA[harm][1] * t + paramA[harm][2])
        sb += paramB[harm][0] * np.sin(paramB[harm][1] * t + paramB[harm][2])
        sc += paramC[harm][0] * np.sin(paramC[harm][1] * t + paramC[harm][2])
    sinA.append(sa)
    sinB.append(sb)
    sinC.append(sc)

In [ ]:
df = pd.DataFrame(columns=["mse", "mae"], index=["a_t", "b_t", "corr"])

df.loc["a_t", "mse"] = np.mean((np.array(sinA) - np.array(test_data_a)) ** 2)
df.loc["a_t", "mae"] = np.mean(np.abs(np.array(sinA) - np.array(test_data_a)))
df.loc["b_t", "mse"] = np.mean((np.array(sinB) - np.array(test_data_b)) ** 2)
df.loc["b_t", "mae"] = np.mean(np.abs(np.array(sinB) - np.array(test_data_b)))
df.loc["corr", "mse"] = np.mean((np.array(sinC) - np.array(test_data_corr)) ** 2)
df.loc["corr", "mae"] = np.mean(np.abs(np.array(sinC) - np.array(test_data_corr)))

df

In [ ]:
fig = go.Figure()
x = gmm["dates"][train_size + 1 : len(correlation)]
# Add traces for the first plot (coef_a)
fig.add_trace(
    go.Scatter(x=x, y=sinA, mode="lines", name="coef_a", line=dict(color=colors[1]))
)
# Add traces for the second plot (coef_b)
fig.add_trace(
    go.Scatter(
        x=x, y=test_data_a, mode="lines", name="coef_b", line=dict(color=colors[0])
    )
)

# Update layout for better presentation (optional)
smooth_title = ""
smooth_file = ""
if smoothcoef:
    smooth_title = f" smoothed by {smooth_N} minutes."
    smooth_file = f"_smooth_{smooth_N}"

mae = df["mae"]["a_t"]
mse = df["mse"]["a_t"]

fig.update_layout(
    height=figure_size["height"],
    width=figure_size["width"],
    showlegend=False,
    title=dict(
        font=dict(size=24),
        text=f"<b>MSE: <i>{mse:.6f}</i>,\t MSA: <i>{mae:.6f}</i></b>",
        x=0.5,
    ),
)

fig.update_xaxes(
    tickmode="linear",
    tickformat="%d-%m",
    tick0=x[0],  # Optional: Starting point anchor for the ticks
    dtick=86400000 * 3,  # Forces a tick every 2 days (2 * 1 day in ms)
    tickangle=45,
    tickvals=convert_dateticks(x),
    tickfont=dict(size=axis_font_size),
    # dtick=pd.Timedelta("1 days").total_seconds() * 1000,
    # tickangle=90,
    title_text="Дата",
    title_font={"size": axis_title_font_size},
)

fig.update_yaxes(
    tickfont=dict(size=axis_font_size),
    title_text="Значение, нТ",
    title_font={"size": axis_title_font_size},
)


fig.write_image(
    f"{out_dir}/d{series_name}_harmonic_forecasting_a_{hn}_{correlation_window[0]}{smooth_file}.png"
)

In [ ]:
fig = go.Figure()
x = gmm["dates"][train_size + 1 : len(correlation)]
# Add traces for the first plot (coef_a)
fig.add_trace(
    go.Scatter(x=x, y=sinB, mode="lines", name="coef_a", line=dict(color=colors[1]))
)
# Add traces for the second plot (coef_b)
fig.add_trace(
    go.Scatter(
        x=x, y=test_data_b, mode="lines", name="coef_b", line=dict(color=colors[0])
    )
)

# Update layout for better presentation (optional)
smooth_title = ""
smooth_file = ""
if smoothcoef:
    smooth_title = f" smoothed by {smooth_N} minutes."
    smooth_file = f"_smooth_{smooth_N}"

mae = df["mae"]["b_t"]
mse = df["mse"]["b_t"]

fig.update_layout(
    height=figure_size["height"],
    width=figure_size["width"],
    showlegend=False,
    title=dict(
        font=dict(size=24),
        text=f"<b>MSE: <i>{mse:.6f}</i>,\t MSA: <i>{mae:.6f}</i></b>",
        x=0.5,
    ),
)

fig.update_xaxes(
    tickmode="linear",
    tickformat="%d-%m",
    tick0=x[0],  # Optional: Starting point anchor for the ticks
    dtick=86400000 * 3,  # Forces a tick every 2 days (2 * 1 day in ms)
    tickangle=45,
    tickvals=convert_dateticks(x),
    tickfont=dict(size=axis_font_size),
    # dtick=pd.Timedelta("1 days").total_seconds() * 1000,
    # tickangle=90,
    title_text="Дата",
    title_font={"size": axis_title_font_size},
)

fig.update_yaxes(
    tickfont=dict(size=axis_font_size),
    title_text="Значение, нТ",
    title_font={"size": axis_title_font_size},
)

fig.write_image(
    f"{out_dir}/d{series_name}_harmonic_forecasting_b_{hn}_{correlation_window[0]}{smooth_file}.png"
)

In [ ]:
fig = go.Figure()
x = gmm["dates"][train_size + 1 : len(correlation)]
# Add traces for the first plot (coef_a)
fig.add_trace(
    go.Scatter(x=x, y=sinC, mode="lines", name="coef_a", line=dict(color=colors[1]))
)
# Add traces for the second plot (coef_b)
fig.add_trace(
    go.Scatter(
        x=x, y=test_data_corr, mode="lines", name="coef_b", line=dict(color=colors[0])
    )
)

# Update layout for better presentation (optional)
smooth_title = ""
smooth_file = ""
if smoothcoef:
    smooth_title = f" smoothed by {smooth_N} minutes."
    smooth_file = f"_smooth_{smooth_N}"

mae = df["mae"]["corr"]
mse = df["mse"]["corr"]

fig.update_layout(
    height=figure_size["height"],
    width=figure_size["width"],
    showlegend=False,
    title=dict(
        font=dict(size=24),
        text=f"<b>MSE: <i>{mse:.6f}</i>,\t MSA: <i>{mae:.6f}</i></b>",
        x=0.5,
    ),
)

fig.update_xaxes(
    tickmode="linear",
    tickformat="%d-%m",
    tick0=x[0],  # Optional: Starting point anchor for the ticks
    dtick=86400000 * 3,  # Forces a tick every 2 days (2 * 1 day in ms)
    tickangle=45,
    tickvals=convert_dateticks(x),
    tickfont=dict(size=axis_font_size),
    # dtick=pd.Timedelta("1 days").total_seconds() * 1000,
    # tickangle=90,
    title_text="Дата",
    title_font={"size": axis_title_font_size},
)

fig.update_yaxes(
    tickfont=dict(size=axis_font_size),
    title_text="Значение, нТ",
    title_font={"size": axis_title_font_size},
)

fig.write_image(
    f"{out_dir}/d{series_name}_harmonic_forecasting_corr_{hn}_{correlation_window[0]}{smooth_file}.png"
)